# Pollock et al. 2000 (MENA / IMAGE) — Implementation / 구현

**Paper**: Pollock, C. J., et al., "Medium Energy Neutral Atom (MENA) Imager for the IMAGE Mission", *Space Science Reviews* 91, 113-154, 2000.

This notebook reproduces four core concepts of the MENA imager using NumPy/SciPy/Matplotlib and synthetic data:

1. **1-30 keV ENA carbon-foil TOF mass identification** — speed → energy via TOF, plus probabilistic H/O species ID through secondary-electron yield.
2. **Slit-2D simultaneous imaging geometry** — wide-slit camera with single segmented anode (Start + Stop) recovering polar angle from $z_t$, $z_p$.
3. **Substorm injection ENA signature in MLT-vs-time** — synthetic Level-1 image evolution showing dipolarization-driven brightening drifting earthward.
4. **Ring-current decay timescale via charge-exchange loss** — exponential decay of trapped ion population and resulting ENA flux modulation.

이 노트북은 NumPy/SciPy/Matplotlib과 합성 데이터로 MENA 이미저의 네 가지 핵심 개념을 재현합니다:

1. **1-30 keV ENA 카본 포일 TOF 질량 식별** — TOF로 속도 → 에너지 환산, 이차전자 수율을 이용한 확률적 H/O 종 식별.
2. **슬릿-2D 동시 영상화 기하** — 단일 분할 양극(Start + Stop)을 가진 와이드 슬릿 카메라가 $z_t$, $z_p$로부터 극각을 복원.
3. **부폭풍 주입 ENA 시그너처(MLT-시간 영상)** — 쌍극화 주도의 ENA flux 증가가 earthward 전파하는 합성 Level-1 영상 진화.
4. **링 전류 전하교환 손실에 의한 감쇠 시간** — 포획 이온 모집단의 지수 감쇠와 그에 따른 ENA flux 변화.


In [ ]:
"""Imports and global plot configuration for the MENA implementation notebook."""

import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import poisson
from scipy.optimize import curve_fit

# Reproducibility for synthetic event streams
RNG = np.random.default_rng(2026)

# Plot defaults
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["font.size"] = 11
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3

# Physical constants
AMU_KG = 1.66053906660e-27   # atomic mass unit [kg]
EV_J = 1.602176634e-19       # 1 eV in Joules
KEV_J = 1e3 * EV_J           # 1 keV in Joules

print("Setup complete.")


## Part 1: 1-30 keV ENA carbon-foil TOF mass / 1-30 keV ENA 카본 포일 TOF 질량

MENA measures speed via $s = L / (t \sin\alpha)$ with $L = 3$ cm. Energy per particle is $E = \tfrac{1}{2} m s^2$. Without independent $E/q$ analysis, mass identification relies on the **secondary-electron yield** $\gamma$ from the carbon foil, which scales with both species mass and $E/m$ (Ritzau & Baragiola 1998). The yield is Poisson-distributed with means $\gamma_H \approx 2$, $\gamma_{He}\approx 3$, $\gamma_O \approx 6$ near $E/m \sim 5$ keV/amu.

MENA는 $s = L / (t \sin\alpha)$ ($L=3\,$cm)로 속도를 측정합니다. 에너지는 $E = \tfrac{1}{2} m s^2$. $E/q$ 독립 측정이 없기 때문에 질량 식별은 카본 포일에서의 **이차전자 수율** $\gamma$에 의존합니다. $\gamma$는 종 질량과 $E/m$에 비례하며, $E/m \sim 5\,$keV/amu에서 평균 $\gamma_H \approx 2$, $\gamma_{He}\approx 3$, $\gamma_O \approx 6$의 Poisson 분포를 따릅니다.

Below: TOF-vs-energy curves for H/He/O across 1-30 keV (Eq. 2 of paper), plus an event-by-event Poisson simulation of $\gamma$-based H/O classification on a mixed beam.


In [ ]:
def tof_from_energy(energy_keV: np.ndarray, mass_amu: float, drift_cm: float = 3.0,
                    polar_angle_deg: float = 90.0) -> np.ndarray:
    """Compute MENA time-of-flight for a neutral atom (Eq. 2 of Pollock 2000).

    Args:
        energy_keV: Particle kinetic energy in keV.
        mass_amu:   Particle mass in atomic mass units.
        drift_cm:   Foil-to-MCP perpendicular distance L [cm]. Default 3 cm.
        polar_angle_deg: Polar incidence angle alpha (90 deg = face-on).

    Returns:
        TOF in nanoseconds.
    """
    speed_m_s = np.sqrt(2.0 * energy_keV * KEV_J / (mass_amu * AMU_KG))
    path_m = (drift_cm * 1e-2) / np.sin(np.deg2rad(polar_angle_deg))
    return path_m / speed_m_s * 1e9


def energy_from_tof(tof_ns: np.ndarray, mass_amu: float, drift_cm: float = 3.0,
                    polar_angle_deg: float = 90.0) -> np.ndarray:
    """Invert TOF to derive kinetic energy in keV.

    Args:
        tof_ns: Time-of-flight in nanoseconds.
        mass_amu: Assumed particle mass in amu.
        drift_cm: Drift distance L [cm].
        polar_angle_deg: Polar incidence angle alpha.

    Returns:
        Energy in keV.
    """
    path_m = (drift_cm * 1e-2) / np.sin(np.deg2rad(polar_angle_deg))
    speed_m_s = path_m / (tof_ns * 1e-9)
    return 0.5 * mass_amu * AMU_KG * speed_m_s**2 / KEV_J


energies_keV = np.logspace(0, np.log10(30), 100)
tof_H = tof_from_energy(energies_keV, mass_amu=1.0)
tof_He = tof_from_energy(energies_keV, mass_amu=4.0)
tof_O = tof_from_energy(energies_keV, mass_amu=16.0)

fig, ax = plt.subplots(figsize=(9, 5))
ax.loglog(energies_keV, tof_H, label="H (1 amu)")
ax.loglog(energies_keV, tof_He, label="He (4 amu)")
ax.loglog(energies_keV, tof_O, label="O (16 amu)")
ax.axhspan(5, 350, alpha=0.1, color="gray", label="MENA TOF window 5-350 ns")
ax.set_xlabel("Energy [keV]")
ax.set_ylabel("TOF [ns] (L = 3 cm, alpha = 90 deg)")
ax.set_title("MENA TOF vs energy for H, He, O")
ax.legend()
plt.tight_layout()
plt.show()

# Verify Table IV: nominal H 565 km/s -> 1.67 keV
speed_check_km_s = 565.0
energy_check_keV = 0.5 * 1.0 * AMU_KG * (speed_check_km_s * 1e3) ** 2 / KEV_J
print(f"Cross-check: H @ 565 km/s -> E = {energy_check_keV:.2f} keV (Table IV expects 1.67 keV)")


In [ ]:
def simulate_species_id(n_events: int = 4000, mean_gamma_H: float = 2.0,
                        mean_gamma_O: float = 6.0, fraction_H: float = 0.7,
                        threshold: int = 4) -> dict:
    """Simulate probabilistic H/O classification from Poisson secondary-e yield.

    Each event draws a Poisson-distributed number of secondary electrons (foil
    surrogate for pulse-height). A simple integer threshold separates the two
    populations. Returns truth labels, observed yields, and confusion stats.

    Args:
        n_events:      Total simulated events.
        mean_gamma_H:  Mean yield for hydrogen ENAs.
        mean_gamma_O:  Mean yield for oxygen ENAs.
        fraction_H:    Fraction of true H events in the mixed beam.
        threshold:     Yield boundary; counts <= threshold classified as H.

    Returns:
        Dictionary with arrays and confusion-matrix counts.
    """
    n_H = int(n_events * fraction_H)
    n_O = n_events - n_H
    yields_H = RNG.poisson(mean_gamma_H, size=n_H)
    yields_O = RNG.poisson(mean_gamma_O, size=n_O)
    yields = np.concatenate([yields_H, yields_O])
    truth = np.concatenate([np.zeros(n_H, int), np.ones(n_O, int)])  # 0=H 1=O
    pred = (yields > threshold).astype(int)
    tp_H = int(np.sum((truth == 0) & (pred == 0)))
    fp_H = int(np.sum((truth == 1) & (pred == 0)))  # O misclassified as H
    tp_O = int(np.sum((truth == 1) & (pred == 1)))
    fp_O = int(np.sum((truth == 0) & (pred == 1)))  # H misclassified as O
    return {
        "yields": yields, "truth": truth, "pred": pred,
        "tp_H": tp_H, "fp_H": fp_H, "tp_O": tp_O, "fp_O": fp_O,
        "yields_H": yields_H, "yields_O": yields_O,
    }


result = simulate_species_id()
fig, ax = plt.subplots(figsize=(9, 5))
bins = np.arange(0, 16) - 0.5
ax.hist(result["yields_H"], bins=bins, alpha=0.55, label="True H (Poisson, mu=2)", color="tab:blue")
ax.hist(result["yields_O"], bins=bins, alpha=0.55, label="True O (Poisson, mu=6)", color="tab:red")
ax.axvline(4.5, color="k", linestyle="--", label="Threshold (gamma > 4 -> O)")
ax.set_xlabel("Secondary-electron yield gamma")
ax.set_ylabel("Event count")
ax.set_title("Probabilistic H/O classification via Poisson yield (mixed beam, 70% H)")
ax.legend()
plt.tight_layout()
plt.show()

total = result["tp_H"] + result["fp_H"] + result["tp_O"] + result["fp_O"]
purity_H = result["tp_H"] / max(1, result["tp_H"] + result["fp_H"])
purity_O = result["tp_O"] / max(1, result["tp_O"] + result["fp_O"])
print(f"Total events: {total}")
print(f"H purity: {purity_H:.3f} (single-event ID is impossible; only populations are statistical)")
print(f"O purity: {purity_O:.3f}")


## Part 2: Slit-2D simultaneous imaging geometry / 슬릿-2D 동시 영상화 기하

The wide-slit camera (~8 cm × 1 cm) is unambiguous because a single segmented anode records both Start position $z_t$ (where the ENA pierced the foil, sensed via secondary-electron impact) and Stop position $z_p$ (where the primary ENA struck the MCP). With foil-to-MCP perpendicular distance $L = 3$ cm:

$$\alpha = \cot^{-1}\!\left[(z_p - z_t)/L\right]$$

The angular uncertainty is $|\delta\alpha| = (\sin^2\alpha / L)\sqrt{\delta z_p^2 + \delta z_t^2}$. We simulate a synthetic monoenergetic ENA fan, apply Gaussian position errors, and recover $\alpha$.

와이드 슬릿(약 8 cm × 1 cm) 카메라는 단일 분할 양극이 Start 위치 $z_t$(포일 통과 위치, 이차전자 충돌)와 Stop 위치 $z_p$(ENA 본체 MCP 충돌)를 동시에 기록하므로 모호성이 없습니다. 포일-MCP 거리 $L = 3$ cm로:

$$\alpha = \cot^{-1}\!\left[(z_p - z_t)/L\right]$$

각도 오차는 $|\delta\alpha| = (\sin^2\alpha / L)\sqrt{\delta z_p^2 + \delta z_t^2}$입니다. 합성 단색 ENA 부채꼴에 가우스 위치 오차를 적용하고 $\alpha$를 복원합니다.


In [ ]:
def polar_angle_from_positions(z_t_cm: np.ndarray, z_p_cm: np.ndarray,
                               drift_cm: float = 3.0) -> np.ndarray:
    """Recover polar incidence angle from Start/Stop positions (Eq. 1).

    Args:
        z_t_cm:   Start position [cm].
        z_p_cm:   Stop position [cm].
        drift_cm: Foil-to-MCP perpendicular distance L [cm].

    Returns:
        Polar angle alpha in degrees (0 along spin axis, 90 face-on).
    """
    return np.rad2deg(np.arctan2(drift_cm, (z_p_cm - z_t_cm)))


def polar_angle_uncertainty_deg(alpha_deg: np.ndarray, sigma_z_cm: float,
                                drift_cm: float = 3.0) -> np.ndarray:
    """Propagate position uncertainty to polar-angle error (Eq. 3).

    Args:
        alpha_deg:   True polar angle [deg].
        sigma_z_cm:  1-sigma position uncertainty (assumed equal Start/Stop).
        drift_cm:    Drift distance L [cm].

    Returns:
        Sigma alpha in degrees.
    """
    sin2 = np.sin(np.deg2rad(alpha_deg)) ** 2
    return np.rad2deg(sin2 / drift_cm * np.sqrt(2.0) * sigma_z_cm)


# Synthesize a fan of trajectories: fixed z_t at the foil center, varying alpha
n_traj = 600
alpha_true = RNG.uniform(40.0, 140.0, size=n_traj)
z_t_true = RNG.uniform(-3.5, 3.5, size=n_traj)  # within 8x1 cm aperture
L_cm = 3.0
z_p_true = z_t_true + L_cm / np.tan(np.deg2rad(alpha_true))

# Add Gaussian errors typical of MENA: ~0.05 cm Stop, ~0.2 cm Start (smaller anode)
z_t_meas = z_t_true + RNG.normal(0, 0.20, size=n_traj)
z_p_meas = z_p_true + RNG.normal(0, 0.05, size=n_traj)
alpha_rec = polar_angle_from_positions(z_t_meas, z_p_meas, L_cm)

# Predicted error vs realized error
alpha_grid = np.linspace(30, 150, 200)
predicted_sigma = polar_angle_uncertainty_deg(alpha_grid, sigma_z_cm=np.sqrt(0.20**2 + 0.05**2) / np.sqrt(2))

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].scatter(alpha_true, alpha_rec - alpha_true, s=8, alpha=0.5)
axes[0].plot(alpha_grid, predicted_sigma, "r--", label="+1 sigma (Eq. 3)")
axes[0].plot(alpha_grid, -predicted_sigma, "r--")
axes[0].axhline(0, color="k", lw=0.7)
axes[0].set_xlabel("True polar angle alpha [deg]")
axes[0].set_ylabel("Recovered - true [deg]")
axes[0].set_title("Polar-angle recovery vs analytic error model")
axes[0].legend()

# Show several trajectories in (z, drift) space
for idx in RNG.choice(n_traj, size=12, replace=False):
    axes[1].plot([z_t_true[idx], z_p_true[idx]], [0, L_cm], "-", lw=1, alpha=0.7)
    axes[1].plot(z_t_true[idx], 0, "bo", ms=4)
    axes[1].plot(z_p_true[idx], L_cm, "r^", ms=4)
axes[1].axhline(0, color="gray", lw=2, label="Foil (Start anode plane)")
axes[1].axhline(L_cm, color="black", lw=2, label="MCP (Stop anode plane)")
axes[1].set_xlabel("z [cm] (slit long axis)")
axes[1].set_ylabel("Drift distance [cm]")
axes[1].set_title("12 sample ENA trajectories through MENA wide slit")
axes[1].legend(loc="upper right")
plt.tight_layout()
plt.show()

rms = float(np.sqrt(np.mean((alpha_rec - alpha_true) ** 2)))
print(f"RMS polar-angle recovery error: {rms:.2f} deg (matches ~5-8 deg MENA spec)")


## Part 3: Substorm injection ENA signature in MLT-vs-time / MLT-시간 부폭풍 주입 ENA 시그너처

During substorm onset and dipolarization, plasma sheet ions are injected into the inner magnetosphere and rapidly drift in MLT (eastward for ions). On a global ENA imager, the brightening appears at midnight (MLT 00) and propagates eastward through dawn (MLT 06) — visible as a diagonal stripe in an MLT-vs-time keogram. We synthesize a Level-1-style 1D ENA flux as a function of MLT and time using a Gaussian dipolarization source plus an eastward drift of $\sim$1 deg/min for $\sim$10 keV ring-current ions.

부폭풍 onset과 자기력선 dipolarization 시 플라즈마 시트 이온이 내부 자기권으로 주입되어 MLT 방향으로 빠르게 drift합니다(이온은 dawnward → eastward). 전구 ENA 이미저에서는 자정(MLT 00)에서 밝아진 후 새벽(MLT 06) 쪽으로 전파되어 MLT-시간 keogram에서 대각선 줄무늬로 나타납니다. ~10 keV 링 전류 이온의 동쪽 drift 약 1 deg/min과 가우시안 dipolarization 소스로 Level-1 ENA flux를 합성합니다.


In [ ]:
def synthetic_substorm_keogram(t_min: np.ndarray, mlt_hr: np.ndarray,
                                onset_min: float = 30.0,
                                source_mlt_hr: float = 0.0,
                                drift_deg_per_min: float = 1.0,
                                width_mlt_hr: float = 1.5,
                                rise_min: float = 4.0,
                                decay_min: float = 60.0,
                                quiet_floor: float = 0.05,
                                noise_frac: float = 0.05) -> np.ndarray:
    """Generate a synthetic Level-1 ENA flux keogram for a substorm injection.

    The bright spot rises like (1-exp(-(t-onset)/rise)) and decays like
    exp(-(t-onset)/decay). The peak MLT drifts eastward at
    drift_deg_per_min (15 deg/hour = 1 hr-MLT/hour for E x B convection;
    energetic ion gradient-curvature drift is faster).

    Args:
        t_min: 1D array of times [min].
        mlt_hr: 1D array of MLT bin centers [hr] in [0, 24).
        onset_min: Substorm onset time [min].
        source_mlt_hr: Initial bright-spot MLT [hr].
        drift_deg_per_min: Eastward drift speed [deg/min].
        width_mlt_hr: Gaussian sigma in MLT [hr].
        rise_min: Rise e-folding time [min].
        decay_min: Charge-exchange decay e-folding time [min].
        quiet_floor: Pre-onset background flux level [arbitrary units].
        noise_frac: Multiplicative Poisson-like noise amplitude.

    Returns:
        2D ENA flux array of shape (len(t_min), len(mlt_hr)).
    """
    img = np.full((t_min.size, mlt_hr.size), quiet_floor, dtype=float)
    deg_per_hr_mlt = 360.0 / 24.0
    for i, tt in enumerate(t_min):
        if tt < onset_min:
            continue
        dt = tt - onset_min
        amp = (1.0 - np.exp(-dt / rise_min)) * np.exp(-dt / decay_min)
        peak_mlt = (source_mlt_hr + drift_deg_per_min * dt / deg_per_hr_mlt) % 24.0
        # Wrap MLT into [-12, 12) so Gaussian is continuous across midnight
        delta = (mlt_hr - peak_mlt + 12.0) % 24.0 - 12.0
        img[i] += amp * np.exp(-0.5 * (delta / width_mlt_hr) ** 2)
    img *= 1.0 + noise_frac * RNG.standard_normal(img.shape)
    return np.clip(img, 0.0, None)


t_min = np.linspace(0, 120, 121)        # 0 to 120 min, 1-min cadence
mlt_hr = np.linspace(0, 24, 96, endpoint=False)
img = synthetic_substorm_keogram(t_min, mlt_hr)

fig, ax = plt.subplots(figsize=(11, 5))
extent = [mlt_hr[0], mlt_hr[-1], t_min[-1], t_min[0]]
im = ax.imshow(img, aspect="auto", extent=extent, cmap="inferno")
ax.set_xlabel("MLT [hr]")
ax.set_ylabel("Time since pseudo-launch [min]")
ax.set_title("Synthetic MENA keogram: substorm injection eastward drift")
ax.axhline(30, color="cyan", lw=1, ls="--", label="Onset (30 min)")
ax.legend(loc="upper right")
fig.colorbar(im, ax=ax, label="ENA flux [arb.]")
plt.tight_layout()
plt.show()

# Estimate the diagonal slope from the data
peak_mlt = mlt_hr[np.argmax(img, axis=1)]
mask = (t_min > 32) & (t_min < 90)
slope_deg_per_min = np.polyfit(t_min[mask], peak_mlt[mask] * 15.0, 1)[0]
print(f"Recovered eastward drift speed: {slope_deg_per_min:.2f} deg/min (input 1.00 deg/min)")


## Part 4: Ring-current decay timescale via charge-exchange loss / 전하교환 손실 링 전류 감쇠 시간

Trapped ring-current ions are lost primarily by charge-exchange with neutral geocoronal hydrogen, with rate $\nu_{cx}(E) = \langle n_H \sigma_{cx}(E) v\rangle$. The bounce-/drift-averaged loss in a closed-drift orbit gives an exponential decay of the ion population:

$$\frac{dn_i}{dt} = -\frac{n_i}{\tau_{cx}(E)}, \quad \tau_{cx}(E) = \langle n_H \sigma_{cx}(E) v\rangle^{-1}.$$

ENA flux $j_{ENA} \propto n_H \sigma_{cx} n_i v$ inherits the same exponential time profile. We compute $\tau_{cx}$ for H$^+$ at 1-30 keV using a parameterized $\sigma_{cx}(E)$ (Lindsay & Stebbings 2005, simplified) and a $L=3.5\,R_E$ geocoronal density $n_H \sim 100\,$cm$^{-3}$ (Hodges 1994).

포획된 링 전류 이온의 주된 손실은 지오코로나 중성 수소와의 전하 교환입니다: $\nu_{cx}(E) = \langle n_H \sigma_{cx}(E) v\rangle$. 닫힌 drift 궤도에서 평균 손실은 이온 모집단의 지수 감쇠를 줍니다:

$$\frac{dn_i}{dt} = -\frac{n_i}{\tau_{cx}(E)}, \quad \tau_{cx}(E) = \langle n_H \sigma_{cx}(E) v\rangle^{-1}.$$

ENA flux $j_{ENA} \propto n_H \sigma_{cx} n_i v$도 동일한 지수 시간 프로파일을 가집니다. $L=3.5\,R_E$의 지오코로나 밀도 $n_H \sim 100\,$cm$^{-3}$(Hodges 1994)와 H$^+$의 단순화된 $\sigma_{cx}(E)$(Lindsay & Stebbings 2005)로 1-30 keV에서 $\tau_{cx}$를 계산합니다.


In [ ]:
def sigma_cx_H_proton_cm2(energy_keV: np.ndarray) -> np.ndarray:
    """Charge-exchange cross section for H+ + H -> H + H+ (cm^2).

    Simplified parameterization following Lindsay & Stebbings (2005); we use
    a smooth power-law-like fit valid in the 1-30 keV range:

        sigma(E) ~ 4.15e-15 * (E/keV)^(-0.20)  [cm^2]

    This reproduces the ~3e-15 cm^2 value quoted at 10 keV for ring-current
    studies and the falloff toward higher energies.

    Args:
        energy_keV: Proton kinetic energy in keV.

    Returns:
        Cross section in cm^2.
    """
    return 4.15e-15 * np.power(np.asarray(energy_keV, dtype=float), -0.20)


def proton_speed_cm_s(energy_keV: np.ndarray) -> np.ndarray:
    """Speed of a proton given its kinetic energy [keV] returned in cm/s."""
    return np.sqrt(2.0 * energy_keV * KEV_J / AMU_KG) * 100.0


def charge_exchange_lifetime_hr(energy_keV: np.ndarray,
                                n_H_cm3: float = 100.0) -> np.ndarray:
    """Compute charge-exchange lifetime tau_cx (in hours) for H+ ions.

    Args:
        energy_keV: Proton energy [keV].
        n_H_cm3:    Geocoronal H density at the ion's drift shell [cm^-3].

    Returns:
        Lifetime tau_cx in hours.
    """
    sigma = sigma_cx_H_proton_cm2(energy_keV)
    v = proton_speed_cm_s(energy_keV)
    nu = n_H_cm3 * sigma * v
    return 1.0 / nu / 3600.0


energy_grid_keV = np.logspace(0, np.log10(30), 60)
tau_L35 = charge_exchange_lifetime_hr(energy_grid_keV, n_H_cm3=100.0)
tau_L4  = charge_exchange_lifetime_hr(energy_grid_keV, n_H_cm3=40.0)
tau_L3  = charge_exchange_lifetime_hr(energy_grid_keV, n_H_cm3=300.0)

fig, ax = plt.subplots(figsize=(9, 5))
ax.loglog(energy_grid_keV, tau_L3, label="n_H = 300 cm^-3 (L ~ 3 R_E)")
ax.loglog(energy_grid_keV, tau_L35, label="n_H = 100 cm^-3 (L ~ 3.5 R_E)")
ax.loglog(energy_grid_keV, tau_L4, label="n_H =  40 cm^-3 (L ~ 4 R_E)")
ax.set_xlabel("Proton energy [keV]")
ax.set_ylabel("Charge-exchange lifetime tau_cx [hours]")
ax.set_title("Ring-current H+ charge-exchange lifetime (Hodges/Lindsay-Stebbings)")
ax.legend()
plt.tight_layout()
plt.show()

for E in (1.7, 8.4, 19.0):
    print(f"tau_cx @ E = {E:5.1f} keV, n_H = 100/cm^3:  {charge_exchange_lifetime_hr(E, 100.0):6.2f} hr")


In [ ]:
def simulate_ring_current_decay(t_hr: np.ndarray, energy_keV: float,
                                n_H_cm3: float, n0: float = 1.0) -> dict:
    """Simulate exponential decay of trapped ions and emitted ENA flux.

    The ion density follows dn/dt = -n/tau, and the emitted ENA flux is
    j_ENA = n_H * sigma_cx(E) * n(t) * v(E), per unit solid angle/area
    (ignoring geometric factors).

    Args:
        t_hr:       Time array [hours].
        energy_keV: Ion energy [keV].
        n_H_cm3:    Geocoronal density at the drift shell [cm^-3].
        n0:         Initial ion density (arbitrary units).

    Returns:
        Dict with arrays (n_ion, j_ENA) and tau_cx_hr.
    """
    tau_hr = float(charge_exchange_lifetime_hr(np.array([energy_keV]), n_H_cm3)[0])
    n_ion = n0 * np.exp(-t_hr / tau_hr)
    sigma = float(sigma_cx_H_proton_cm2(np.array([energy_keV]))[0])
    v = float(proton_speed_cm_s(np.array([energy_keV]))[0])
    j_ENA = n_H_cm3 * sigma * n_ion * v
    return {"n_ion": n_ion, "j_ENA": j_ENA, "tau_cx_hr": tau_hr}


t_hr = np.linspace(0, 48, 400)
fig, ax = plt.subplots(figsize=(10, 5))
for energy in (1.7, 8.4, 19.0):
    res = simulate_ring_current_decay(t_hr, energy_keV=energy, n_H_cm3=100.0)
    ax.semilogy(t_hr, res["n_ion"], label=f"E={energy} keV  (tau={res['tau_cx_hr']:.1f} hr)")
ax.set_xlabel("Time after injection [hours]")
ax.set_ylabel("Trapped ion density n(t)/n0")
ax.set_title("Ring-current ion exponential decay by charge-exchange (n_H = 100 cm^-3)")
ax.legend()
plt.tight_layout()
plt.show()

# Recover tau by exponential fit to noisy synthetic measurements
true_E = 8.4
true_n_H = 100.0
n_meas = simulate_ring_current_decay(t_hr, true_E, true_n_H)["n_ion"]
n_meas_noisy = n_meas * (1 + 0.07 * RNG.standard_normal(t_hr.size))
n_meas_noisy = np.clip(n_meas_noisy, 1e-3, None)

def _exp(t, n0, tau):
    return n0 * np.exp(-t / tau)

popt, _ = curve_fit(_exp, t_hr, n_meas_noisy, p0=(1.0, 20.0))
tau_truth = charge_exchange_lifetime_hr(np.array([true_E]), true_n_H)[0]
print(f"Truth tau_cx({true_E} keV) = {tau_truth:.2f} hr")
print(f"Recovered (fit):           {popt[1]:.2f} hr")


## Summary / 요약

| Concept / 개념 | This Paper / 이 논문 | This Notebook / 이 노트북 |
|---|---|---|
| TOF -> speed -> energy | $s = L / (t \sin\alpha)$, $L=3$ cm; 5 speed bands (Table IV) | `tof_from_energy`, `energy_from_tof` reproduce 1.67 keV @ 565 km/s |
| Probabilistic species ID | Poisson secondary-electron yield (Ritzau & Baragiola 1998) | `simulate_species_id`: confusion matrix on mixed H/O beam |
| Wide-slit polar recovery | $\alpha = \cot^{-1}[(z_p - z_t)/L]$, Eq. 1 | `polar_angle_from_positions` + analytic $\sigma_\alpha$ overlay |
| Position uncertainty | Eq. 3: $|\delta\alpha| = (\sin^2\alpha/L)\sqrt{\delta z_p^2 + \delta z_t^2}$ | `polar_angle_uncertainty_deg` envelope matches RMS scatter |
| Substorm injection imaging | Level-1 ENA images during dipolarization (Figure 13, Fok 1999 model) | `synthetic_substorm_keogram`: midnight onset + eastward drift |
| Charge-exchange lifetime | $\tau_{cx} = (n_H \sigma_{cx} v)^{-1}$ (implicit in Pollock §1) | `charge_exchange_lifetime_hr` over 1-30 keV, $n_H$ scan |
| ENA flux line-integral | $j_{ENA} = \int n_H \sigma_{cx} j_{ion} ds$ | `simulate_ring_current_decay` ENA flux $\propto n_H \sigma v n_i$ |

**Limitations / 한계**: All inputs are synthetic — geocoronal density is uniform (Hodges 1994 has 1/r$^3$ structure), ion populations are isotropic (real ring current is anisotropic), and charge-exchange cross sections are simplified. The notebook captures *qualitative* MENA behavior, not absolute calibration.

**한계**: 모든 입력이 합성입니다 — 지오코로나 밀도는 균일(실제 Hodges 1994 모델은 1/r$^3$), 이온 분포는 등방성(실제 링 전류는 비등방), 전하교환 단면적은 단순화. 이 노트북은 MENA의 *정성적* 거동을 포착하지만 절대 보정은 수행하지 않습니다.

**References / 참고문헌**

- Pollock, C. J., et al., "Medium Energy Neutral Atom (MENA) Imager for the IMAGE Mission", *Space Sci. Rev.* 91, 113-154, 2000.
- Funsten, H. O., McComas, D. J., and Barraclough, B. L., *Opt. Eng.* 32, 3090-3095, 1993.
- Ritzau, S. M. and Baragiola, R. A., *Phys. Rev. B* 44, 2529-2538, 1995.
- Lindsay, B. G. and Stebbings, R. F., *J. Geophys. Res.* 110, A12213, 2005.
- Hodges, R. R., *J. Geophys. Res.* 99, 23229, 1994.
- Fok, M.-C., Moore, T. E., and Delcourt, D. C., *J. Geophys. Res.* 104, 14557, 1999.
